# Task 2.7 — SFT Backtesting & Base vs SFT Comparison
Full backtesting on SFT-rescored signals with direct comparison to base model results.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, pearsonr, sem

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
TASK1_DIR = (_cwd.parent / "task_1").resolve()
OUTPUT_DIR = _cwd / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SFT_SIGNALS_PATH = OUTPUT_DIR / "filing_signals_sft.jsonl"
BASE_SIGNALS_PATH = OUTPUT_DIR / "filing_signals.jsonl"
RETURNS_PATH = TASK1_DIR / "output" / "filing_returns.csv"

# ── Time splits ──────────────────────────────────────────────────────────
TRAIN = ("2015-01-01", "2019-12-31")
VAL   = ("2020-01-01", "2022-12-31")
TEST  = ("2023-01-01", "2025-12-31")

TRAIN_END = pd.Timestamp("2020-01-01")
VAL_END   = pd.Timestamp("2023-01-01")

# ── Constants ────────────────────────────────────────────────────────────
LABEL_ORDER = ["very_negative", "negative", "neutral", "positive", "very_positive"]
COHORT_COLORS = ["#d62728", "#ff7f0e", "#999999", "#2ca02c", "#1f77b4"]
COHORT_RANK_MAP = {c: i for i, c in enumerate(LABEL_ORDER)}

# ── Helper: score to cohort label ────────────────────────────────────────
def score_to_label(score: float) -> str:
    if score <= -1.0:
        return "very_negative"
    elif score <= -0.33:
        return "negative"
    elif score <= 0.33:
        return "neutral"
    elif score <= 1.0:
        return "positive"
    else:
        return "very_positive"

# ── Helper: assign period ────────────────────────────────────────────────
def assign_period(dt: pd.Timestamp) -> str:
    if dt < TRAIN_END:
        return "train"
    elif dt < VAL_END:
        return "val"
    else:
        return "test"

# ── Load SFT signals ────────────────────────────────────────────────────
sft_rows = []
with open(SFT_SIGNALS_PATH) as f:
    for line in f:
        sft_rows.append(json.loads(line))
sft_df = pd.DataFrame(sft_rows)
sft_df["filing_date"] = pd.to_datetime(sft_df["filing_date"])
sft_df["period"] = sft_df["filing_date"].apply(assign_period)
sft_df["cohort"] = pd.Categorical(sft_df["cohort"], categories=LABEL_ORDER, ordered=True)
sft_df["cohort_rank"] = sft_df["cohort"].map(COHORT_RANK_MAP)

# ── Load Base signals ───────────────────────────────────────────────────
base_rows = []
with open(BASE_SIGNALS_PATH) as f:
    for line in f:
        base_rows.append(json.loads(line))
base_df = pd.DataFrame(base_rows)
base_df["filing_date"] = pd.to_datetime(base_df["filing_date"])
base_df["period"] = base_df["filing_date"].apply(assign_period)
base_df["cohort"] = pd.Categorical(base_df["cohort"], categories=LABEL_ORDER, ordered=True)
base_df["cohort_rank"] = base_df["cohort"].map(COHORT_RANK_MAP)

# ── Summary ──────────────────────────────────────────────────────────────
print(f"SFT signals loaded: {len(sft_df):,} filings")
print(f"Base signals loaded: {len(base_df):,} filings")
print(f"\nSFT period split:")
for p in ["train", "val", "test"]:
    n = (sft_df["period"] == p).sum()
    dr = sft_df.loc[sft_df["period"] == p, "filing_date"]
    if n > 0:
        print(f"  {p:6s}: {n:5d} filings  ({dr.min().strftime('%Y-%m')} to {dr.max().strftime('%Y-%m')})")
    else:
        print(f"  {p:6s}: {n:5d} filings")

print(f"\nBase period split:")
for p in ["train", "val", "test"]:
    n = (base_df["period"] == p).sum()
    dr = base_df.loc[base_df["period"] == p, "filing_date"]
    if n > 0:
        print(f"  {p:6s}: {n:5d} filings  ({dr.min().strftime('%Y-%m')} to {dr.max().strftime('%Y-%m')})")
    else:
        print(f"  {p:6s}: {n:5d} filings")

## Per-Category Returns by Cohort — TRAIN (SFT)
Expand `category_scores` into long format, assign cohort labels, compute mean 21-day excess return per (category, cohort) pair in the training period (2015-2019).

In [ ]:
# ── Expand category_scores into long format for SFT ─────────────────────
def expand_category_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Expand category_scores dict column into one row per (filing, category)."""
    rows = []
    for _, row in df.iterrows():
        cat_scores = row["category_scores"]
        if not isinstance(cat_scores, dict):
            continue
        for cat, score in cat_scores.items():
            rows.append({
                "ticker": row["ticker"],
                "filing_date": row["filing_date"],
                "sub_sector": row["sub_sector"],
                "period": row["period"],
                "ret_21d_excess": row["ret_21d_excess"],
                "category": cat,
                "cat_score": score,
                "cat_cohort": score_to_label(score),
            })
    return pd.DataFrame(rows)

sft_cat_df = expand_category_scores(sft_df)
sft_cat_df["cat_cohort"] = pd.Categorical(
    sft_cat_df["cat_cohort"], categories=LABEL_ORDER, ordered=True
)
sft_cat_df["year_month"] = sft_cat_df["filing_date"].dt.to_period("M")

print(f"SFT filing-category pairs: {len(sft_cat_df):,}")
print(f"Categories: {sft_cat_df['category'].nunique()}")

# ── TRAIN period: per-category returns by cohort ────────────────────────
train_cat = sft_cat_df[sft_cat_df["period"] == "train"]
n_train = train_cat.groupby(["ticker", "filing_date"]).ngroups

# Mean return per (category, cohort)
mean_tbl = train_cat.pivot_table(
    index="category", columns="cat_cohort", values="ret_21d_excess",
    aggfunc="mean", observed=False,
).reindex(columns=LABEL_ORDER) * 100

# Average row
avg_row = mean_tbl.mean(axis=0)
avg_row.name = "__AVG_ALL_CATEGORIES__"
mean_tbl = pd.concat([mean_tbl, avg_row.to_frame().T])

# Count per (category, cohort)
count_tbl = train_cat.pivot_table(
    index="category", columns="cat_cohort", values="ret_21d_excess",
    aggfunc="count", observed=False,
).reindex(columns=LABEL_ORDER).fillna(0).astype(int)

total_row = count_tbl.sum(axis=0)
total_row.name = "__TOTAL__"
count_tbl = pd.concat([count_tbl, total_row.to_frame().T])

print(f"\n{'=' * 90}")
print(f"  TRAIN — Mean 21-Day Excess Return by Category & Sentiment Cohort ({n_train} filings)")
print(f"{'=' * 90}")

fmt_mean = mean_tbl.copy()
for col in fmt_mean.columns:
    fmt_mean[col] = fmt_mean[col].apply(lambda x: f"{x:.2f}%" if pd.notna(x) else "--")
print(fmt_mean.to_string())

print(f"\nTRAIN — Observation Counts")
print("-" * 90)
print(count_tbl.to_string())

## Per-Category Returns by Cohort — VAL (SFT)
Validation period (2020-2022).

In [ ]:
# ── VAL period: per-category returns by cohort ──────────────────────────
val_cat = sft_cat_df[sft_cat_df["period"] == "val"]
n_val = val_cat.groupby(["ticker", "filing_date"]).ngroups

mean_tbl_val = val_cat.pivot_table(
    index="category", columns="cat_cohort", values="ret_21d_excess",
    aggfunc="mean", observed=False,
).reindex(columns=LABEL_ORDER) * 100

avg_row_val = mean_tbl_val.mean(axis=0)
avg_row_val.name = "__AVG_ALL_CATEGORIES__"
mean_tbl_val = pd.concat([mean_tbl_val, avg_row_val.to_frame().T])

count_tbl_val = val_cat.pivot_table(
    index="category", columns="cat_cohort", values="ret_21d_excess",
    aggfunc="count", observed=False,
).reindex(columns=LABEL_ORDER).fillna(0).astype(int)

total_row_val = count_tbl_val.sum(axis=0)
total_row_val.name = "__TOTAL__"
count_tbl_val = pd.concat([count_tbl_val, total_row_val.to_frame().T])

print(f"{'=' * 90}")
print(f"  VAL — Mean 21-Day Excess Return by Category & Sentiment Cohort ({n_val} filings)")
print(f"{'=' * 90}")

fmt_val = mean_tbl_val.copy()
for col in fmt_val.columns:
    fmt_val[col] = fmt_val[col].apply(lambda x: f"{x:.2f}%" if pd.notna(x) else "--")
print(fmt_val.to_string())

print(f"\nVAL — Observation Counts")
print("-" * 90)
print(count_tbl_val.to_string())

## Per-Category Returns by Cohort — TEST (SFT)
Test period (2023-2025).

In [ ]:
# ── TEST period: per-category returns by cohort ─────────────────────────
test_cat = sft_cat_df[sft_cat_df["period"] == "test"]
n_test = test_cat.groupby(["ticker", "filing_date"]).ngroups

mean_tbl_test = test_cat.pivot_table(
    index="category", columns="cat_cohort", values="ret_21d_excess",
    aggfunc="mean", observed=False,
).reindex(columns=LABEL_ORDER) * 100

avg_row_test = mean_tbl_test.mean(axis=0)
avg_row_test.name = "__AVG_ALL_CATEGORIES__"
mean_tbl_test = pd.concat([mean_tbl_test, avg_row_test.to_frame().T])

count_tbl_test = test_cat.pivot_table(
    index="category", columns="cat_cohort", values="ret_21d_excess",
    aggfunc="count", observed=False,
).reindex(columns=LABEL_ORDER).fillna(0).astype(int)

total_row_test = count_tbl_test.sum(axis=0)
total_row_test.name = "__TOTAL__"
count_tbl_test = pd.concat([count_tbl_test, total_row_test.to_frame().T])

print(f"{'=' * 90}")
print(f"  TEST — Mean 21-Day Excess Return by Category & Sentiment Cohort ({n_test} filings)")
print(f"{'=' * 90}")

fmt_test = mean_tbl_test.copy()
for col in fmt_test.columns:
    fmt_test[col] = fmt_test[col].apply(lambda x: f"{x:.2f}%" if pd.notna(x) else "--")
print(fmt_test.to_string())

print(f"\nTEST — Observation Counts")
print("-" * 90)
print(count_tbl_test.to_string())

## Factor Ranking — IS vs OOS (SFT)
Monthly long-short returns per category. IS = train (2015-2019), OOS = val+test (2020-2025).
- Long = mean excess return of filings with positive category score
- Short = mean excess return of filings with negative category score
- Monthly L/S = long - short

In [ ]:
# ── Compute factor return: sign(cat_score) * ret_21d_excess ─────────────
fc = sft_cat_df.copy()
fc = fc[fc["cat_score"] != 0].copy()
fc["factor_return"] = np.sign(fc["cat_score"]) * fc["ret_21d_excess"]

# Monthly aggregation per category
monthly = (
    fc.groupby(["category", "year_month", "period"])["factor_return"]
    .mean()
    .reset_index()
)

# ── Stats computation ───────────────────────────────────────────────────
def compute_ls_stats(df: pd.DataFrame) -> pd.Series:
    """Compute Sharpe, t-stat, mean, std for monthly factor returns."""
    vals = df["factor_return"].values
    n = len(vals)
    if n < 2:
        return pd.Series({
            "mean (%)": np.nan, "std (%)": np.nan,
            "Sharpe (ann.)": np.nan, "tstat": np.nan, "n_months": n,
        })
    m = vals.mean()
    s = vals.std(ddof=1)
    sharpe = m / s * np.sqrt(12) if s > 0 else np.nan
    t_stat = m / (s / np.sqrt(n)) if s > 0 else np.nan
    return pd.Series({
        "mean (%)": m * 100,
        "std (%)": s * 100,
        "Sharpe (ann.)": sharpe,
        "tstat": t_stat,
        "n_months": n,
    })

is_monthly = monthly[monthly["period"] == "train"]
oos_monthly = monthly[monthly["period"].isin(["val", "test"])]

is_stats = (
    is_monthly.groupby("category")
    .apply(compute_ls_stats, include_groups=False)
    .sort_values("Sharpe (ann.)", ascending=False)
)
oos_stats = (
    oos_monthly.groupby("category")
    .apply(compute_ls_stats, include_groups=False)
    .sort_values("Sharpe (ann.)", ascending=False)
)

print("SFT Factor Ranking — IS (Train 2015-2019)")
print("=" * 85)
print(is_stats.to_string(float_format="{:.3f}".format))

print(f"\nSFT Factor Ranking — OOS (Val+Test 2020-2025)")
print("=" * 85)
oos_reordered = oos_stats.reindex(is_stats.index)
print(oos_reordered.to_string(float_format="{:.3f}".format))

# Top 5 by IS Sharpe
top5_categories = is_stats.index[:5].tolist()
print(f"\nTop 5 categories by IS Sharpe: {top5_categories}")

## Cumulative Returns — Top 5 Categories (SFT)
Monthly cumulative L/S return for top 5 categories by IS Sharpe, plus equal-weight average.

In [ ]:
sns.set_style("whitegrid")

# Build monthly return series per category (full timeline)
monthly_returns = monthly.pivot_table(
    index="year_month", columns="category", values="factor_return", aggfunc="mean"
)
monthly_returns = monthly_returns.sort_index()

# Equal-weight average
monthly_returns["__EW_AVG__"] = monthly_returns.mean(axis=1)

# Convert to timestamps for plotting
plot_idx = monthly_returns.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(top5_categories)))
for i, cat in enumerate(top5_categories):
    if cat in monthly_returns.columns:
        cum_ret = monthly_returns[cat].fillna(0).cumsum() * 100
        ax.plot(plot_idx, cum_ret, label=cat, color=colors[i], linewidth=1.5)

# EW average
cum_ew = monthly_returns["__EW_AVG__"].fillna(0).cumsum() * 100
ax.plot(plot_idx, cum_ew, label="Equal-Weight Avg", color="black",
        linewidth=2, linestyle="--")

# Period boundaries
ax.axvline(TRAIN_END, color="red", linestyle=":", alpha=0.7, label="Train/Val split")
ax.axvline(VAL_END, color="blue", linestyle=":", alpha=0.7, label="Val/Test split")

ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Cumulative Return (%)", fontsize=11)
ax.set_title("SFT — Factor Cumulative Return (Top 5 by IS Sharpe)", fontsize=13)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## Rolling t-statistics (SFT)
12-month and 24-month rolling windows for top 5 categories. Horizontal lines at +/-1.96.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
windows = [12, 24]
colors = plt.cm.tab10(np.linspace(0, 1, len(top5_categories)))

for ax_idx, window in enumerate(windows):
    ax = axes[ax_idx]
    for i, cat in enumerate(top5_categories):
        if cat not in monthly_returns.columns:
            continue
        series = monthly_returns[cat].fillna(0)
        rolling_mean = series.rolling(window).mean()
        rolling_std = series.rolling(window).std()
        rolling_t = rolling_mean / (rolling_std / np.sqrt(window))
        ax.plot(plot_idx, rolling_t, label=cat, color=colors[i], linewidth=1.2)

    ax.axhline(1.96, color="gray", linestyle="--", alpha=0.5, linewidth=0.8)
    ax.axhline(-1.96, color="gray", linestyle="--", alpha=0.5, linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(TRAIN_END, color="red", linestyle=":", alpha=0.7)
    ax.axvline(VAL_END, color="blue", linestyle=":", alpha=0.7)
    ax.set_ylabel(f"t-stat ({window}m rolling)", fontsize=10)
    ax.set_title(f"Rolling {window}-Month t-statistic", fontsize=12)
    ax.legend(fontsize=7, loc="upper left", ncol=2)

axes[1].set_xlabel("Date", fontsize=11)
fig.suptitle("SFT — Rolling t-statistics (Top 5 by IS Sharpe)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Cohort vs 21-Day Excess Return (SFT) --- The Key Chart
Filing-level cohort vs mean 21-day excess return. Bar chart with error bars, Spearman correlation, long-short spread, and Information Ratio.

In [ ]:
# ── Cohort return stats (SFT) ────────────────────────────────────────────
cohort_stats_sft = (
    sft_df.groupby("cohort", observed=False)["ret_21d_excess"]
    .agg(["mean", "std", "count", "sem"])
    .reindex(LABEL_ORDER)
)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(LABEL_ORDER))
means = cohort_stats_sft["mean"].values
sems = cohort_stats_sft["sem"].values

bars = ax.bar(x, means * 100, yerr=sems * 100, capsize=5,
              color=COHORT_COLORS, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=10)
ax.set_ylabel("Mean 21-Day Excess Return (%)", fontsize=11)
ax.set_title("SFT — Cohort vs. Mean 21-Day Excess Return", fontsize=13)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

for bar, m, s, n in zip(bars, means, sems, cohort_stats_sft["count"].values):
    label_y = (bar.get_height() + s * 100 + 0.05) if m >= 0 else (bar.get_height() - s * 100 - 0.15)
    ax.text(bar.get_x() + bar.get_width() / 2, label_y,
            f"{m*100:.2f}%\n(n={n})", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

# ── Spearman: cohort rank vs return ──────────────────────────────────────
rho_sft, pval_sft = spearmanr(sft_df["cohort_rank"], sft_df["ret_21d_excess"])
print(f"Spearman(cohort_rank, ret_21d_excess): rho={rho_sft:.4f}, p={pval_sft:.4e}")

# ── Long-short spread ───────────────────────────────────────────────────
vp_mask = sft_df["cohort"] == "very_positive"
vn_mask = sft_df["cohort"] == "very_negative"
long_ret = sft_df.loc[vp_mask, "ret_21d_excess"].mean() if vp_mask.any() else np.nan
short_ret = sft_df.loc[vn_mask, "ret_21d_excess"].mean() if vn_mask.any() else np.nan
ls_spread_sft = long_ret - short_ret
print(f"Long-short spread (VP - VN): {ls_spread_sft*100:.3f}%")

# ── Information Ratio ────────────────────────────────────────────────────
mean_excess_sft = sft_df["ret_21d_excess"].mean()
std_excess_sft = sft_df["ret_21d_excess"].std()
ir_sft = mean_excess_sft / std_excess_sft if std_excess_sft > 0 else np.nan
print(f"Information Ratio: {ir_sft:.4f}")

print(f"\nCohort summary table:")
print(cohort_stats_sft.to_string(float_format="{:.4f}".format))

## Signal vs 21-Day Excess Return Scatter (SFT)
Continuous signal vs return, colored by sub-sector, with OLS fit line and R-squared.

In [ ]:
sector_palette = {
    "airlines": "#e41a1c",
    "defense": "#377eb8",
    "industrial_equipment": "#4daf4a",
    "general": "#984ea3",
}

fig, ax = plt.subplots(figsize=(10, 7))

for sector, color in sector_palette.items():
    mask = sft_df["sub_sector"] == sector
    ax.scatter(
        sft_df.loc[mask, "signal"],
        sft_df.loc[mask, "ret_21d_excess"] * 100,
        alpha=0.3, s=15, c=color, label=sector,
    )

# OLS fit line
z = np.polyfit(sft_df["signal"], sft_df["ret_21d_excess"] * 100, 1)
p = np.poly1d(z)
x_range = np.linspace(sft_df["signal"].min(), sft_df["signal"].max(), 100)
ax.plot(x_range, p(x_range), "k--", linewidth=2, label="OLS fit")

# R-squared
ss_res = np.sum((sft_df["ret_21d_excess"] * 100 - p(sft_df["signal"])) ** 2)
ss_tot = np.sum((sft_df["ret_21d_excess"] * 100 - sft_df["ret_21d_excess"].mean() * 100) ** 2)
r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
ax.text(0.05, 0.95, f"$R^2$ = {r_squared:.4f}",
        transform=ax.transAxes, fontsize=11, va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

ax.set_xlabel("Filing Signal", fontsize=11)
ax.set_ylabel("21-Day Excess Return (%)", fontsize=11)
ax.set_title("SFT — Signal vs. 21-Day Excess Return", fontsize=13)
ax.legend(fontsize=9)
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)

plt.tight_layout()
plt.show()

# Correlations
rho_s, p_s = spearmanr(sft_df["signal"], sft_df["ret_21d_excess"])
rho_p, p_p = pearsonr(sft_df["signal"], sft_df["ret_21d_excess"])
print(f"Spearman: rho={rho_s:.4f}, p={p_s:.4e}")
print(f"Pearson:  r={rho_p:.4f}, p={p_p:.4e}")

## Per-Sector Cohort vs Return (SFT)
4-panel breakdown: airlines, defense, general, industrial_equipment. Each panel shows cohort bars with error bars and Spearman annotation.

In [ ]:
sectors = sorted(sft_df["sub_sector"].unique())
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
axes = axes.flatten()

for i, sector in enumerate(sectors[:4]):
    ax = axes[i]
    sector_data = sft_df[sft_df["sub_sector"] == sector]

    sector_cohort = (
        sector_data.groupby("cohort", observed=False)["ret_21d_excess"]
        .agg(["mean", "sem", "count"])
        .reindex(LABEL_ORDER)
    )

    x = np.arange(len(LABEL_ORDER))
    means_s = sector_cohort["mean"].fillna(0).values
    sems_s = sector_cohort["sem"].fillna(0).values
    counts_s = sector_cohort["count"].fillna(0).astype(int).values

    bars = ax.bar(x, means_s * 100, yerr=sems_s * 100, capsize=4,
                  color=COHORT_COLORS, edgecolor="black", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=8)
    ax.set_title(f"{sector} (n={len(sector_data)})", fontsize=11)
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")

    for bar, n in zip(bars, counts_s):
        if n > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, 0.02,
                    f"n={n}", ha="center", va="bottom", fontsize=7, color="gray")

    # Spearman per sector
    if len(sector_data) >= 5:
        rho_sec, p_sec = spearmanr(sector_data["cohort_rank"], sector_data["ret_21d_excess"])
        ax.text(0.02, 0.95, f"Spearman={rho_sec:.3f}\n(p={p_sec:.3f})",
                transform=ax.transAxes, fontsize=8, va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# Handle case where fewer than 4 sectors exist
for j in range(len(sectors), 4):
    axes[j].set_visible(False)

fig.supylabel("Mean 21-Day Excess Return (%)", fontsize=11)
fig.suptitle("SFT — Cohort vs. Return by Sub-Sector", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## BASE vs SFT COMPARISON --- Side by Side
The central comparison: does SFT-rescoring produce better investment signals than the base model?

**12a:** Cohort distribution comparison
**12b:** Cohort return comparison
**12c:** Monotonicity comparison (Spearman)
**12d:** Key metrics table

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BASE vs SFT COMPARISON
# ═══════════════════════════════════════════════════════════════════════════

# ── Compute base cohort stats ────────────────────────────────────────────
cohort_stats_base = (
    base_df.groupby("cohort", observed=False)["ret_21d_excess"]
    .agg(["mean", "std", "count", "sem"])
    .reindex(LABEL_ORDER)
)

# ── Base Spearman & long-short ───────────────────────────────────────────
base_df["cohort_rank"] = base_df["cohort"].map(COHORT_RANK_MAP)
rho_base, pval_base = spearmanr(base_df["cohort_rank"], base_df["ret_21d_excess"])

vp_base = base_df.loc[base_df["cohort"] == "very_positive", "ret_21d_excess"].mean()
vn_base = base_df.loc[base_df["cohort"] == "very_negative", "ret_21d_excess"].mean()
ls_spread_base = vp_base - vn_base

ir_base = base_df["ret_21d_excess"].mean() / base_df["ret_21d_excess"].std()

# ── Also compute base category-level stats for IS/OOS comparison ─────────
base_cat_df = expand_category_scores(base_df)
base_cat_df["cat_cohort"] = pd.Categorical(
    base_cat_df["cat_cohort"], categories=LABEL_ORDER, ordered=True
)
base_cat_df["year_month"] = base_cat_df["filing_date"].dt.to_period("M")

fc_base = base_cat_df[base_cat_df["cat_score"] != 0].copy()
fc_base["factor_return"] = np.sign(fc_base["cat_score"]) * fc_base["ret_21d_excess"]

monthly_base = (
    fc_base.groupby(["category", "year_month", "period"])["factor_return"]
    .mean()
    .reset_index()
)

is_base = monthly_base[monthly_base["period"] == "train"]
oos_base = monthly_base[monthly_base["period"].isin(["val", "test"])]

is_stats_base = (
    is_base.groupby("category")
    .apply(compute_ls_stats, include_groups=False)
    .sort_values("Sharpe (ann.)", ascending=False)
)
oos_stats_base = (
    oos_base.groupby("category")
    .apply(compute_ls_stats, include_groups=False)
    .sort_values("Sharpe (ann.)", ascending=False)
)

# ═══════════════════════════════════════════════════════════════════════════
# 12a: Cohort Distribution Comparison
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, df, title, color in [
    (axes[0], base_df, "Base Model", "#d62728"),
    (axes[1], sft_df, "SFT Model", "#1f77b4"),
]:
    counts = df["cohort"].value_counts().reindex(LABEL_ORDER).fillna(0).astype(int)
    x = np.arange(len(LABEL_ORDER))
    bars = ax.bar(x, counts.values, color=COHORT_COLORS, edgecolor="black", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=9)
    ax.set_title(f"{title} (n={len(df)})", fontsize=12)
    ax.set_ylabel("Number of Filings", fontsize=10)

    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                str(int(val)), ha="center", va="bottom", fontsize=10, fontweight="bold")

fig.suptitle("Cohort Distribution — Base vs SFT", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# 12b: Cohort Return Comparison (grouped bar chart)
# ═══════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(LABEL_ORDER))
width = 0.35

means_base = cohort_stats_base["mean"].values * 100
sems_base = cohort_stats_base["sem"].values * 100
means_sft = cohort_stats_sft["mean"].values * 100
sems_sft = cohort_stats_sft["sem"].values * 100

bars_base = ax.bar(x - width / 2, means_base, width, yerr=sems_base, capsize=4,
                   label="Base", color="#d62728", alpha=0.7, edgecolor="black", linewidth=0.5)
bars_sft = ax.bar(x + width / 2, means_sft, width, yerr=sems_sft, capsize=4,
                  label="SFT", color="#1f77b4", alpha=0.7, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=10)
ax.set_ylabel("Mean 21-Day Excess Return (%)", fontsize=11)
ax.set_title("Cohort vs Return — Base vs SFT", fontsize=13)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.legend(fontsize=11)

# Annotate bars
for bars, means_arr, counts_arr in [
    (bars_base, means_base, cohort_stats_base["count"].values),
    (bars_sft, means_sft, cohort_stats_sft["count"].values),
]:
    for bar, m, n in zip(bars, means_arr, counts_arr):
        y_pos = bar.get_height() + 0.05 if m >= 0 else bar.get_height() - 0.15
        ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
                f"{m:.2f}%", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# 12c: Monotonicity Comparison
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("  MONOTONICITY COMPARISON — Base vs SFT")
print("=" * 70)
print(f"\n  Base Model:  Spearman rho = {rho_base:.4f},  p = {pval_base:.4e}")
print(f"  SFT Model:   Spearman rho = {rho_sft:.4f},  p = {pval_sft:.4e}")
print(f"  Delta rho:   {rho_sft - rho_base:+.4f}")

# Check monotonicity of mean returns
base_means = cohort_stats_base["mean"].values
sft_means = cohort_stats_sft["mean"].values

base_monotonic = all(a <= b for a, b in zip(base_means, base_means[1:]))
sft_monotonic = all(a <= b for a, b in zip(sft_means, sft_means[1:]))
print(f"\n  Base monotonic (VN < N < Neut < P < VP): {base_monotonic}")
print(f"  SFT monotonic  (VN < N < Neut < P < VP): {sft_monotonic}")

# Rank correlation of cohort means
base_rank_order = np.argsort(np.argsort(base_means))
sft_rank_order = np.argsort(np.argsort(sft_means))
ideal_order = np.array([0, 1, 2, 3, 4])
base_kendall = np.corrcoef(base_rank_order, ideal_order)[0, 1]
sft_kendall = np.corrcoef(sft_rank_order, ideal_order)[0, 1]
print(f"\n  Base cohort-mean rank correlation with ideal: {base_kendall:.3f}")
print(f"  SFT cohort-mean rank correlation with ideal:  {sft_kendall:.3f}")

# ═══════════════════════════════════════════════════════════════════════════
# 12d: Key Metrics Table
# ═══════════════════════════════════════════════════════════════════════════

# Find top IS category for both base and SFT to report its Sharpe
top_cat_base = is_stats_base.index[0] if len(is_stats_base) > 0 else "N/A"
top_cat_sft = is_stats.index[0] if len(is_stats) > 0 else "N/A"

top_is_sharpe_base = is_stats_base.iloc[0]["Sharpe (ann.)"] if len(is_stats_base) > 0 else np.nan
top_oos_sharpe_base = (
    oos_stats_base.loc[top_cat_base, "Sharpe (ann.)"]
    if top_cat_base in oos_stats_base.index else np.nan
)
top_is_sharpe_sft = is_stats.iloc[0]["Sharpe (ann.)"] if len(is_stats) > 0 else np.nan
top_oos_sharpe_sft = (
    oos_stats.loc[top_cat_sft, "Sharpe (ann.)"]
    if top_cat_sft in oos_stats.index else np.nan
)

metrics = {
    "VP cohort size": (
        int(cohort_stats_base.loc["very_positive", "count"]),
        int(cohort_stats_sft.loc["very_positive", "count"]),
    ),
    "VP mean return (%)": (
        cohort_stats_base.loc["very_positive", "mean"] * 100,
        cohort_stats_sft.loc["very_positive", "mean"] * 100,
    ),
    "VN cohort size": (
        int(cohort_stats_base.loc["very_negative", "count"]),
        int(cohort_stats_sft.loc["very_negative", "count"]),
    ),
    "VN mean return (%)": (
        cohort_stats_base.loc["very_negative", "mean"] * 100,
        cohort_stats_sft.loc["very_negative", "mean"] * 100,
    ),
    "Long-short spread (%)": (
        ls_spread_base * 100,
        ls_spread_sft * 100,
    ),
    "Spearman rho": (rho_base, rho_sft),
    "p-value": (pval_base, pval_sft),
    "Information Ratio": (ir_base, ir_sft),
    f"Top cat IS Sharpe ({top_cat_base} / {top_cat_sft})": (
        top_is_sharpe_base, top_is_sharpe_sft,
    ),
    f"Top cat OOS Sharpe ({top_cat_base} / {top_cat_sft})": (
        top_oos_sharpe_base, top_oos_sharpe_sft,
    ),
}

print("\n" + "=" * 70)
print("  KEY METRICS COMPARISON — Base vs SFT")
print("=" * 70)
print(f"\n{'Metric':45s}  {'Base':>10s}  {'SFT':>10s}  {'Delta':>10s}")
print("-" * 80)

comparison_records = []
for metric_name, (base_val, sft_val) in metrics.items():
    delta = sft_val - base_val if not (np.isnan(base_val) or np.isnan(sft_val)) else np.nan
    comparison_records.append({
        "metric": metric_name, "base": base_val, "sft": sft_val, "delta": delta,
    })

    # Format for display
    if "size" in metric_name:
        print(f"{metric_name:45s}  {int(base_val):10d}  {int(sft_val):10d}  {int(delta):+10d}")
    elif "p-value" in metric_name:
        print(f"{metric_name:45s}  {base_val:10.4e}  {sft_val:10.4e}  {delta:+10.4e}")
    else:
        print(f"{metric_name:45s}  {base_val:10.4f}  {sft_val:10.4f}  {delta:+10.4f}")

print("=" * 70)

## Save Results
Persist SFT backtest results and Base vs SFT comparison to JSON.

In [ ]:
# ── SFT backtest results ─────────────────────────────────────────────────
sft_results = {
    "metadata": {
        "model": "SFT (LoRA fine-tuned)",
        "n_filings": len(sft_df),
        "date_range": [
            sft_df["filing_date"].min().strftime("%Y-%m-%d"),
            sft_df["filing_date"].max().strftime("%Y-%m-%d"),
        ],
        "n_categories": int(sft_cat_df["category"].nunique()),
    },
    "time_split": {
        "train": f"{TRAIN[0]} to {TRAIN[1]}",
        "val": f"{VAL[0]} to {VAL[1]}",
        "test": f"{TEST[0]} to {TEST[1]}",
    },
    "cohort_stats": {
        cohort: {
            "count": int(cohort_stats_sft.loc[cohort, "count"]),
            "mean_return_pct": float(cohort_stats_sft.loc[cohort, "mean"] * 100),
            "std_pct": float(cohort_stats_sft.loc[cohort, "std"] * 100),
        }
        for cohort in LABEL_ORDER
        if cohort in cohort_stats_sft.index
    },
    "signal_metrics": {
        "spearman_rho": float(rho_sft),
        "spearman_pval": float(pval_sft),
        "long_short_spread_pct": float(ls_spread_sft * 100),
        "information_ratio": float(ir_sft),
    },
    "is_ranking": is_stats.reset_index().to_dict(orient="records"),
    "oos_ranking": oos_stats.reset_index().to_dict(orient="records"),
    "top5_categories": top5_categories,
}

sft_results_path = OUTPUT_DIR / "sft_backtest_results.json"
with open(sft_results_path, "w") as f:
    json.dump(sft_results, f, indent=2, default=str)
print(f"Saved: {sft_results_path} ({sft_results_path.stat().st_size / 1024:.1f} KB)")

# ── Base vs SFT comparison ──────────────────────────────────────────────
comparison_output = {
    "metadata": {
        "base_n_filings": len(base_df),
        "sft_n_filings": len(sft_df),
    },
    "metrics": comparison_records,
    "monotonicity": {
        "base_spearman": float(rho_base),
        "sft_spearman": float(rho_sft),
        "base_monotonic": bool(base_monotonic),
        "sft_monotonic": bool(sft_monotonic),
        "base_mean_rank_corr": float(base_kendall),
        "sft_mean_rank_corr": float(sft_kendall),
    },
    "cohort_distributions": {
        "base": {
            c: int(base_df["cohort"].value_counts().reindex(LABEL_ORDER).fillna(0).loc[c])
            for c in LABEL_ORDER
        },
        "sft": {
            c: int(sft_df["cohort"].value_counts().reindex(LABEL_ORDER).fillna(0).loc[c])
            for c in LABEL_ORDER
        },
    },
}

comparison_path = OUTPUT_DIR / "base_vs_sft_comparison.json"
with open(comparison_path, "w") as f:
    json.dump(comparison_output, f, indent=2, default=str)
print(f"Saved: {comparison_path} ({comparison_path.stat().st_size / 1024:.1f} KB)")

# ── Final summary ────────────────────────────────────────────────────────
print(f"\n{'=' * 70}")
print(f"  SFT BACKTESTING COMPLETE")
print(f"{'=' * 70}")
print(f"\n  SFT Spearman rho:      {rho_sft:.4f} (p={pval_sft:.4e})")
print(f"  Base Spearman rho:     {rho_base:.4f} (p={pval_base:.4e})")
print(f"  SFT L/S spread:        {ls_spread_sft*100:+.3f}%")
print(f"  Base L/S spread:       {ls_spread_base*100:+.3f}%")
print(f"  SFT top IS category:   {top_cat_sft} (Sharpe={top_is_sharpe_sft:.2f})")
print(f"  Base top IS category:  {top_cat_base} (Sharpe={top_is_sharpe_base:.2f})")
print(f"\n{'=' * 70}")